# Homework 1 Solution

This notebook shows one possible solution path for Homework 1.

The repository includes a lightweight WDI subset for demonstration. Replace it with the full WDI file if your assignment requires a country that is not included here.

In [1]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


PosixPath('/Users/hanyu/Documents/GitHub/big-data-analysis-2025')

In [2]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "homework_01"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

PosixPath('/Users/hanyu/Documents/GitHub/big-data-analysis-2025/outputs/homework_01')

# Q1. Filter the data for the country you need.

In [3]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "wdi" / "WDI_course_subset.csv")

country_name = "United States"
df_country = df[df["Country Name"] == country_name]

df_country.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
34868,United States,USA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN,NaN
34869,United States,USA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN,NaN
34870,United States,USA,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN,NaN
34871,United States,USA,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN
34872,United States,USA,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN


# Q2. How many rows and columns does the selected country dataset have?

In [4]:
df_country.shape

(1516, 69)

# Q3. Reshape the data into panel format.

In [5]:
panel_data = df_country.drop(columns="Indicator Code").melt(
    id_vars=["Country Name", "Country Code", "Indicator Name"],
    var_name="Year",
).pivot_table(
    values="value",
    index=["Country Name", "Country Code", "Year"],
    columns="Indicator Name",
).reset_index().rename_axis("", axis=1)

panel_data.head()

,Country Name,Country Code,Year,Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),...,"Vulnerable employment, female (% of female employment) (modeled ILO estimate)","Vulnerable employment, male (% of male employment) (modeled ILO estimate)","Vulnerable employment, total (% of total employment) (modeled ILO estimate)","Wage and salaried workers, female (% of female employment) (modeled ILO estimate)","Wage and salaried workers, male (% of male employment) (modeled ILO estimate)","Wage and salaried workers, total (% of total employment) (modeled ILO estimate)","Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)",Women Business and the Law Index Score (scale 1-100),Women's share of population ages 15+ living with HIV (%),Young people (ages 15-24) newly infected with HIV
0,United States,USA,1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,United States,USA,1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,United States,USA,1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,United States,USA,1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,United States,USA,1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Recheck the data types.
panel_data.dtypes

# Convert the original string-like Year column to integer.
panel_data["Year"] = panel_data["Year"].astype(str).astype(int)
panel_data.dtypes


Country Name                                                                                         object
Country Code                                                                                         object
Year                                                                                                  int64
Access to clean fuels and technologies for cooking (% of population)                                float64
Access to clean fuels and technologies for cooking, rural (% of rural population)                   float64
                                                                                                     ...   
Wage and salaried workers, total (% of total employment) (modeled ILO estimate)                     float64
Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)    float64
Women Business and the Law Index Score (scale 1-100)                                                float64
Women's share of population

# Q4. Export the panel data.

In [7]:
panel_data.to_csv(MODULE_OUTPUT_DIR / "Homework1_panel_data.csv", index=False)

# Q5. How many missing values does each variable have?

In [8]:
isna_data = panel_data.isna().sum().sort_values(ascending=True)
isna_data


Country Name                                                                                                                   0
Population ages 0-14 (% of total population)                                                                                   0
Exports of goods and services (current LCU)                                                                                    0
General government final consumption expenditure (% of GDP)                                                                    0
Exports of goods and services (% of GDP)                                                                                       0
                                                                                                                              ..
Completeness of birth registration, female (%)                                                                                64
Completeness of birth registration, male (%)                                                    

# Bonus / Q6. Select the 10 indicators with the fewest missing values and create a compact panel dataset.

This is only one possible answer.

In [10]:
id_vars = ['Country Name', 'Country Code', 'Year']

top_10_features = (
    isna_data.drop(labels=id_vars, errors='ignore')
    .head(10)
    .index
    .tolist()
)

compact_panel = panel_data.loc[:,id_vars + top_10_features ]
compact_panel.head()


,Country Name,Country Code,Year,Population ages 0-14 (% of total population),Exports of goods and services (current LCU),General government final consumption expenditure (% of GDP),Exports of goods and services (% of GDP),General government final consumption expenditure (current LCU),General government final consumption expenditure (current US$),Gross capital formation (% of GDP),Gross capital formation (current LCU),Gross capital formation (current US$),Broad money (% of GDP)
0,United States,USA,1960,31.265168,2.704500e+10,15.582316,4.986260,8.451700e+10,8.451700e+10,22.581702,1.224810e+11,1.224810e+11,60.148365
1,United States,USA,1961,31.404860,2.760200e+10,15.878693,4.905794,8.934000e+10,8.934000e+10,22.479885,1.264810e+11,1.264810e+11,62.735055
2,United States,USA,1962,31.291220,2.906600e+10,16.154326,4.806445,9.769000e+10,9.769000e+10,23.080397,1.395740e+11,1.395740e+11,63.583289
3,United States,USA,1963,31.111261,3.107400e+10,16.096190,4.867627,1.027550e+11,1.027550e+11,23.140104,1.477220e+11,1.477220e+11,65.843450
4,United States,USA,1964,30.933644,3.501900e+10,15.806010,5.107553,1.083710e+11,1.083710e+11,23.123497,1.585420e+11,1.585420e+11,66.842749
